# Stromal/Vascular Reintegration v1.1 FIXED
**Author:** r2end | **Date:** 2026-03-03

## Fixes
- [1] Critical Step11: removed redundant SCANVI.setup_anndata
- [2] Perf Step10: vectorized kNN purity
- [3] Memory Step11: del model_scvi before scANVI train
- [4] Minor Step7: counts fallback from .raw not log1p .X


## Imports + Environment + Configuration + Annotation Table

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Stromal/Vascular Reintegration: Contamination Removal -> scVI -> Tissue-aware scANVI
======================================================================================
Author: r2end  |  Date: 2026-03-03  |  Version: v1.1 (FIXED)

Fixes vs v1.0:
  [1] CRITICAL  Step 11 -- Remove redundant SCANVI.setup_anndata before from_scvi_model
  [2] PERF      Step 10 -- Vectorized kNN purity (eliminates Python for loop)
  [3] MEMORY    Step 11 -- Release model_scvi from GPU before scANVI training
  [4] MINOR     Step 7  -- counts fallback recovers from .raw, not from log1p .X

DROP clusters (contamination, non-myofibroblast):
  Endothelia_vascular_Cap_a_c0          -> Interferon-activated Immune Contaminants
  Endothelia_vascular_venous_systemic_c3 -> MHC-II-high APC-like Contaminants
  Endothelia_vascular_venous_systemic_c5 -> Plasma/B-cell Contaminants
  Fibro_myofibroblast_c0                -> Osteogenic Contaminants
  (Fibro_myofibroblast_c1 KEPT: Unresolved Myofibroblasts)

REASSIGN clusters:
  Endothelia_vascular_venous_systemic_c4 -> L2 = Fibroblasts
  Fibro_adventitial_c2                   -> L2 = Alveolar Fibroblasts
"""

import os
for _k in ["OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"]:
    os.environ[_k] = "8"
import matplotlib; matplotlib.use('Agg')
import warnings; warnings.filterwarnings('ignore')
import gc, time, numpy as np, pandas as pd
import importlib
import sympy

# cell1 bugfix: some environments load sympy without exposing `sympy.printing`,
# which breaks torch/lightning/scvi import chain.
try:
    if not hasattr(sympy, 'printing'):
        sympy.printing = importlib.import_module('sympy.printing')
except Exception as e:
    raise RuntimeError(
        f"Sympy import is broken (sympy module path: {getattr(sympy, '__file__', 'unknown')}). "
        "Please ensure real sympy package is available and no local sympy.py shadows it."
    ) from e

import scanpy as sc
import scvi
import torch
import matplotlib.pyplot as plt
from scipy import sparse
from pathlib import Path
from sklearn.neighbors import NearestNeighbors

PIPELINE_START = time.time()

# ============================================================================
# CONFIGURATION  --  UPDATE TISSUE_KEY / LUNG_TRACHEA_VALUES BEFORE RUNNING
# ============================================================================

INPUT_H5AD  = Path("/home/h2048/data/py/0120/stromal_analysis_unified/results/"
                   "subcluster_unified_v2_20260128/"
                   "adata_stromal_subclustered_FINAL_v2_20260128.h5ad")
OUTPUT_DIR  = Path("/home/h2048/data/py/0303/stromal_reintegration_v1")
FIG_DIR     = OUTPUT_DIR / "figures"
MODEL_DIR   = OUTPUT_DIR / "models"
OUTPUT_H5AD = OUTPUT_DIR / "stromal_reintegrated_scvi_scanvi_v1_1.h5ad"
for d in [OUTPUT_DIR, FIG_DIR, MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)

CELLTYPE_L2 = 'cell_type_L2'
CELLTYPE_L3 = 'cell_type_L3'
BATCH_KEY   = 'sample'

# UPDATE: exact column name and values for lung/trachea tissue
TISSUE_KEY  = 'tissue'
LUNG_TRACHEA_VALUES = {'lung','trachea','Lung','Trachea',
                       'Lung tissue','Tracheal tissue',
                       'lung_tissue','trachea_tissue'}

N_HVG=4000; N_LATENT=75; N_HIDDEN=128; N_LAYERS=2; DROPOUT=0.1
MAX_EPOCHS_SCVI=400; MAX_EPOCHS_SCANVI=200; BATCH_SIZE=256
RANDOM_SEED=42; UNLABELED='Unknown'; MIN_CELLS_BATCH=3
PURITY_K=30; PURITY_THRESHOLD=0.5

# ============================================================================
# REPRODUCIBILITY
# ============================================================================
np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_SEED)
scvi.settings.seed=RANDOM_SEED; scvi.settings.dl_num_workers=0
USE_GPU = torch.cuda.is_available()
sc.settings.verbosity=2; sc.settings.n_jobs=16

# ============================================================================
# ANNOTATION TABLE
# Source: subcluster analysis 2026-02-14, r2end
#
# action: KEEP | REVIEW | DROP | REASSIGN_TO_FIBRO | REASSIGN_TO_ALVEOLAR
#
# Note on Fibro_myofibroblast_c0:
#   coarse_L3 = "Osteogenic Contaminants" -> action=DROP
#   Fibro_myofibroblast_c1 (Unresolved Myofibroblasts) -> action=REVIEW (KEPT)
# ============================================================================

ANNOTATION_TABLE = [
    # --- Lymphatic Endothelium ---
    ("Endothelia_Lymphatic_c0","Lymphatic Endothelia","Quiescent Lymphatic Endothelia","High","KEEP"),
    ("Endothelia_Lymphatic_c1","Lymphatic Endothelia","Venous-like Lymphatic Endothelia","Medium","REVIEW"),
    ("Endothelia_Lymphatic_c2","Lymphatic Endothelia","Quiescent Lymphatic Endothelia","High","KEEP"),
    # --- Capillary aCap ---
    ("Endothelia_vascular_Cap_a_c0","Contaminants","Interferon-activated Immune Contaminants","High","DROP"),
    ("Endothelia_vascular_Cap_a_c1","Capillary Endothelia","Quiescent Aerocyte-like Capillary Endothelia","High","KEEP"),
    ("Endothelia_vascular_Cap_a_c2","Capillary Endothelia","Quiescent Capillary Endothelia","Medium","KEEP"),
    # --- Capillary gCap ---
    ("Endothelia_vascular_Cap_g_c0","Capillary Endothelia","Unresolved Capillary Endothelia","Low","REVIEW"),
    ("Endothelia_vascular_Cap_g_c1","Capillary Endothelia","Interferon-primed Antigen-presenting Capillary Endothelia","Medium","KEEP"),
    ("Endothelia_vascular_Cap_g_c2","Capillary Endothelia","Unresolved Capillary Endothelia","Low","REVIEW"),
    ("Endothelia_vascular_Cap_g_c3","Capillary Endothelia","LYVE1+ Immune-interacting Capillary Endothelia","Medium","REVIEW"),
    # --- Arterial pulmonary ---
    ("Endothelia_vascular_arterial_pulmonary_c0","Arterial Endothelia","Quiescent Arterial Endothelia","High","KEEP"),
    ("Endothelia_vascular_arterial_pulmonary_c1","Arterial Endothelia","Immunomodulatory Arterial Endothelia","High","KEEP"),
    ("Endothelia_vascular_arterial_pulmonary_c2","Arterial Endothelia","Unresolved Arterial Endothelia","Low","REVIEW"),
    # --- Arterial systemic ---
    ("Endothelia_vascular_arterial_systemic_c0","Arterial Endothelia","Quiescent Arterial Endothelia","High","KEEP"),
    ("Endothelia_vascular_arterial_systemic_c1","Arterial Endothelia","Unresolved Arterial Endothelia","Low","REVIEW"),
    # --- Venous pulmonary ---
    ("Endothelia_vascular_venous_pulmonary_c0","Venous Endothelia","Homeostatic Venous Endothelia","Medium","KEEP"),
    ("Endothelia_vascular_venous_pulmonary_c1","Venous Endothelia","Unresolved Venous Endothelia","Low","REVIEW"),
    ("Endothelia_vascular_venous_pulmonary_c2","Venous Endothelia","Unresolved Venous Endothelia","Low","REVIEW"),
    # --- Venous systemic ---
    ("Endothelia_vascular_venous_systemic_c0","Venous Endothelia","Activated Antigen-presenting Venous Endothelia","Medium","REVIEW"),
    ("Endothelia_vascular_venous_systemic_c1","Venous Endothelia","Immune-recruiting (ACKR1+SELE+) Venous Endothelia","High","KEEP"),
    ("Endothelia_vascular_venous_systemic_c2","Venous Endothelia","Angiogenic Venous Endothelia","High","KEEP"),
    ("Endothelia_vascular_venous_systemic_c3","Contaminants","MHC-II-high APC-like Contaminants","High","DROP"),
    ("Endothelia_vascular_venous_systemic_c4","Fibroblasts","Stress-activated Fibroblasts","Medium","REASSIGN_TO_FIBRO"),
    ("Endothelia_vascular_venous_systemic_c5","Contaminants","Plasma/B-cell Contaminants","High","DROP"),
    # --- Fibroblast adventitial ---
    ("Fibro_adventitial_c0","Adventitial Fibroblasts","Perivascular Adventitial Fibroblasts","High","KEEP"),
    ("Fibro_adventitial_c1","Adventitial Fibroblasts","Niche-supporting Adventitial Fibroblasts","High","KEEP"),
    ("Fibro_adventitial_c2","Alveolar Fibroblasts","Homeostatic Alveolar Fibroblasts","Medium","REASSIGN_TO_ALVEOLAR"),
    # --- Fibroblast alveolar ---
    ("Fibro_alveolar_c0","Alveolar Fibroblasts","Unresolved Alveolar Fibroblasts","Low","REVIEW"),
    ("Fibro_alveolar_c1","Alveolar Fibroblasts","Activated ECM-high Alveolar Fibroblasts","High","KEEP"),
    ("Fibro_alveolar_c2","Alveolar Fibroblasts","Homeostatic Alveolar Fibroblasts","High","KEEP"),
    # --- Fibroblast myofibroblast ---
    ("Fibro_myofibroblast_c0","Contaminants","Osteogenic Contaminants","High","DROP"),        # osteogenic != myofibroblast
    ("Fibro_myofibroblast_c1","Myofibroblasts","Unresolved Myofibroblasts","Low","REVIEW"),  # KEPT per user instruction
    # --- Fibroblast peribronchial ---
    ("Fibro_peribronchial_c0","Peribronchial Fibroblasts","Lipogenic Peribronchial Fibroblasts","High","KEEP"),
    ("Fibro_peribronchial_c1","Peribronchial Fibroblasts","Developmental-signaling Peribronchial Fibroblasts","High","KEEP"),
    ("Fibro_peribronchial_c2","Neural/Glial Stromal","Neural-like Stromal Cells","Medium","REVIEW"),
    # --- Pericyte pulmonary ---
    ("Muscle_pericyte_pulmonary_c0","Pericytes","Quiescent Pulmonary Pericytes","High","KEEP"),
    ("Muscle_pericyte_pulmonary_c1","Pericytes","Unresolved Pulmonary Pericytes","Low","REVIEW"),
    ("Muscle_pericyte_pulmonary_c2","Pericytes","Contractile Pulmonary Pericytes","High","KEEP"),
    ("Muscle_pericyte_pulmonary_c3","Pericytes","Precursor Pulmonary Pericytes","Medium","KEEP"),
    # --- Pericyte systemic ---
    ("Muscle_pericyte_systemic_c0","Pericytes","Quiescent Systemic Pericytes","High","KEEP"),
    ("Muscle_pericyte_systemic_c1","Pericytes","Contractile Systemic Pericytes","High","KEEP"),
    # --- Perivascular immune-recruiting ---
    ("Muscle_perivascular_immune_recruiting_c0","Vascular SMC","Contractile Vascular Smooth Muscle Cells","High","KEEP"),
    ("Muscle_perivascular_immune_recruiting_c1","Pericytes","Quiescent Systemic Pericytes","Medium","REVIEW"),
    # --- Smooth muscle ---
    ("Muscle_smooth_arterial_systemic_c0","Airway SMC","Contractile Airway Smooth Muscle Cells","High","KEEP"),
    ("Muscle_smooth_arterial_systemic_c1","Vascular SMC","Contractile Vascular Smooth Muscle Cells","High","KEEP"),
    ("Muscle_smooth_pulmonary_c0","Smooth Muscle Cells","Unresolved Smooth Muscle Cells","Low","REVIEW"),
    ("Muscle_smooth_pulmonary_c1","Smooth Muscle Cells","Contractile Smooth Muscle Cells","High","KEEP"),
    # --- Schwann ---
    ("Schwann_nonmyelinating_c0","Schwann Cells","Non-myelinating Schwann Cells","High","KEEP"),
]

annot_df = pd.DataFrame(ANNOTATION_TABLE,
    columns=['cluster','L2_new','coarse_L3','confidence','action']
).set_index('cluster')

DROP_CLUSTERS = annot_df[annot_df['action']=='DROP'].index.tolist()

# Clusters whose L3 key name contains 'pulmonary' or 'alveolar':
# these are subject to tissue-aware Unknown assignment for non-lung/trachea cells
PULMONARY_ALVEOLAR_CLUSTERS = annot_df[
    annot_df.index.str.contains('pulmonary|alveolar', case=False, regex=True)
] .index.tolist()

print(f"\nDROP clusters ({len(DROP_CLUSTERS)}):")
for c in DROP_CLUSTERS: print(f"  {c} -> [{annot_df.loc[c,'coarse_L3']}]")
print(f"\nPulmonary/Alveolar clusters for tissue-aware scANVI ({len(PULMONARY_ALVEOLAR_CLUSTERS)}):")
for c in PULMONARY_ALVEOLAR_CLUSTERS: print(f"  {c}")



Seed set to 42



DROP clusters (4):
  Endothelia_vascular_Cap_a_c0 -> [Interferon-activated Immune Contaminants]
  Endothelia_vascular_venous_systemic_c3 -> [MHC-II-high APC-like Contaminants]
  Endothelia_vascular_venous_systemic_c5 -> [Plasma/B-cell Contaminants]
  Fibro_myofibroblast_c0 -> [Osteogenic Contaminants]

Pulmonary/Alveolar clusters for tissue-aware scANVI (15):
  Endothelia_vascular_arterial_pulmonary_c0
  Endothelia_vascular_arterial_pulmonary_c1
  Endothelia_vascular_arterial_pulmonary_c2
  Endothelia_vascular_venous_pulmonary_c0
  Endothelia_vascular_venous_pulmonary_c1
  Endothelia_vascular_venous_pulmonary_c2
  Fibro_alveolar_c0
  Fibro_alveolar_c1
  Fibro_alveolar_c2
  Muscle_pericyte_pulmonary_c0
  Muscle_pericyte_pulmonary_c1
  Muscle_pericyte_pulmonary_c2
  Muscle_pericyte_pulmonary_c3
  Muscle_smooth_pulmonary_c0
  Muscle_smooth_pulmonary_c1


## Step 1: LOAD DATA

In [27]:
# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================
print("\n" + "="*70 + "\nSTEP 1: LOAD DATA\n" + "="*70)
adata = sc.read_h5ad(INPUT_H5AD)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"Obs keys: {list(adata.obs.columns)}")

for col in [CELLTYPE_L2, BATCH_KEY]:
    assert col in adata.obs.columns, f"[ERROR] Missing: {col}"
assert CELLTYPE_L3 in adata.obs.columns, f"[ERROR] Missing configured L3 column: {CELLTYPE_L3}"

# Validate whether configured L3 column actually matches annotation table.
# Some source objects store fine subcluster IDs in split form:
#   cell_type_L2 = parent lineage prefix
#   subcluster_id = numeric suffix reused within each lineage
# In that case, reconstruct `ParentType_cN` to match ANNOTATION_TABLE.
def _annotation_match_count(values):
    return pd.Index(values).isin(annot_df.index).sum()

l3_str = adata.obs[CELLTYPE_L3].astype(str)
l3_unique = l3_str.unique()
n_match = _annotation_match_count(l3_unique)

best_col = CELLTYPE_L3
best_unique = l3_unique
best_match = n_match

candidate_cols = []
if 'subcluster_id' in adata.obs.columns:
    candidate_cols.append('subcluster_id')
candidate_cols.extend([
    c for c in adata.obs.columns
    if c != CELLTYPE_L3 and ('l3' in c.lower() or 'subcluster' in c.lower())
])

for c in dict.fromkeys(candidate_cols):
    vals = adata.obs[c].astype(str).unique()
    m = _annotation_match_count(vals)
    if m > best_match:
        best_col = c
        best_unique = vals
        best_match = m

# Fallback: reconstruct fine cluster IDs from L2 prefix + subcluster_id suffix
if best_match == 0 and CELLTYPE_L2 in adata.obs.columns and 'subcluster_id' in adata.obs.columns:
    reconstructed_col = '__reconstructed_cluster_id'
    reconstructed = (
        adata.obs[CELLTYPE_L2].astype(str)
        + '_c'
        + adata.obs['subcluster_id'].astype(str)
    )
    adata.obs[reconstructed_col] = pd.Categorical(reconstructed)
    vals = adata.obs[reconstructed_col].astype(str).unique()
    m = _annotation_match_count(vals)
    if m > best_match:
        print(
            f"[INFO] Reconstructed cluster IDs from '{CELLTYPE_L2}' + 'subcluster_id' "
            f"-> '{reconstructed_col}' (annotation coverage {m}/{len(vals)})"
        )
        best_col = reconstructed_col
        best_unique = vals
        best_match = m

if best_col != CELLTYPE_L3 and best_match > 0:
    print(
        f"[INFO] Auto-switch CELLTYPE_L3: '{CELLTYPE_L3}' -> '{best_col}' "
        f"(annotation coverage {best_match}/{len(best_unique)})"
    )
    CELLTYPE_L3 = best_col
    l3_unique = best_unique
    n_match = best_match
else:
    n_match = best_match

TISSUE_KEY_AVAILABLE = TISSUE_KEY in adata.obs.columns
if not TISSUE_KEY_AVAILABLE:
    similar = [c for c in adata.obs.columns if any(k in c.lower() for k in ('tissue','organ','site'))]
    print(f"[WARN] TISSUE_KEY='{TISSUE_KEY}' not found. Candidates: {similar}")
    print(f"       Tissue-aware Unknown assignment will be SKIPPED.")
else:
    print(f"\nTissue distribution:\n{adata.obs[TISSUE_KEY].value_counts().to_string()}")

not_in_table = [x for x in l3_unique if x not in annot_df.index]
print(f"Annotation table coverage: {len(l3_unique)-len(not_in_table)}/{len(l3_unique)}")
if not_in_table:
    print(f"  [WARN] Unmatched: {not_in_table[:20]}")
if n_match == 0:
    raise ValueError(
        f"[ERROR] No values in '{CELLTYPE_L3}' match ANNOTATION_TABLE. "
        "Check whether fine cluster IDs need reconstruction or update CELLTYPE_L3."
    )




STEP 1: LOAD DATA
Loaded: 60,828 cells x 36,789 genes
Obs keys: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'plateID', 'status', 'donorID', 'cDate', 'age', 'sex', 'cellType', 'percent.mt', 'percent.ribo', 'tissue', 'tissue_sampling_method', 'dataset', 'sample', 'percent.rb', 'decontX_contamination', 'decontX_clusters', 'nCount_decontXcounts', 'nFeature_decontXcounts', 'donor_id', 'Group', 'Ethnicity_inferred', 'Smoker', 'COVID_status', 'First_symptoms_collection_interval', 'Kit_version', 'batch', 'log1p_n_genes', 'percent_total_sarscov2', 'n_counts_sarscov2', 'scrublet_score', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'Source', 'Location', 'CellType', 'BroadCellType', 'organism_ontology_term_id', 'BMI', 'age_or_mean

## Step 2: ATTACH ANNOTATION METADATA

In [28]:
# ============================================================================
# STEP 2: ATTACH ANNOTATION METADATA
# ============================================================================
print("\n" + "="*70 + "\nSTEP 2: ATTACH ANNOTATION TABLE TO adata.obs\n" + "="*70)
l3_str = adata.obs[CELLTYPE_L3].astype(str)
coarse_mapped = l3_str.map(annot_df['coarse_L3'])
action_mapped = l3_str.map(annot_df['action'])
confidence_mapped = l3_str.map(annot_df['confidence'])

n_mapped = coarse_mapped.notna().sum()
print(f"Annotation-mapped cells: {n_mapped:,}/{adata.n_obs:,}")
if n_mapped == 0:
    raise ValueError(
        f"[ERROR] Annotation mapping produced 0 matched cells from '{CELLTYPE_L3}'. "
        "Stopping here to avoid silent all-'Unresolved' downstream labels."
    )

adata.obs['coarse_L3']        = coarse_mapped.fillna('Unresolved')
adata.obs['annot_action']     = action_mapped.fillna('REVIEW')
adata.obs['annot_confidence'] = confidence_mapped.fillna('Low')
print(adata.obs['annot_action'].value_counts().to_string())

if n_mapped < adata.n_obs:
    unmatched = pd.Index(l3_str[coarse_mapped.isna()].unique()).tolist()
    print(f"[WARN] Unmatched {CELLTYPE_L3} values (first 20): {unmatched[:20]}")




STEP 2: ATTACH ANNOTATION TABLE TO adata.obs
Annotation-mapped cells: 60,828/60,828
annot_action
KEEP                    32717
REVIEW                  18742
DROP                     4134
REASSIGN_TO_ALVEOLAR     3558
REASSIGN_TO_FIBRO        1677


## Step 3: REASSIGN MISCLASSIFIED CLUSTERS

In [29]:
# ============================================================================
# STEP 3: REASSIGN MISCLASSIFIED CLUSTERS
# ============================================================================
print("\n" + "="*70 + "\nSTEP 3: REASSIGN MISCLASSIFIED CLUSTERS\n" + "="*70)

# Recompute from current adata to avoid stale state dependency on previous cells
l3_str = adata.obs[CELLTYPE_L3].astype(str)

# If L2 is categorical, ensure target labels exist before assignment
if isinstance(adata.obs[CELLTYPE_L2].dtype, pd.CategoricalDtype):
    target_labels = ['Fibroblasts', 'Alveolar Fibroblasts']
    missing_cats = [x for x in target_labels if x not in adata.obs[CELLTYPE_L2].cat.categories]
    if missing_cats:
        adata.obs[CELLTYPE_L2] = adata.obs[CELLTYPE_L2].cat.add_categories(missing_cats)

mask_fibro = l3_str == 'Endothelia_vascular_venous_systemic_c4'
adata.obs.loc[mask_fibro, CELLTYPE_L2] = 'Fibroblasts'
print(f"REASSIGN_TO_FIBRO: {mask_fibro.sum():,} cells -> L2='Fibroblasts'")

mask_alv = l3_str == 'Fibro_adventitial_c2'
adata.obs.loc[mask_alv, CELLTYPE_L2] = 'Alveolar Fibroblasts'
print(f"REASSIGN_TO_ALVEOLAR: {mask_alv.sum():,} cells -> L2='Alveolar Fibroblasts'")




STEP 3: REASSIGN MISCLASSIFIED CLUSTERS
REASSIGN_TO_FIBRO: 1,677 cells -> L2='Fibroblasts'
REASSIGN_TO_ALVEOLAR: 3,558 cells -> L2='Alveolar Fibroblasts'


## Step 4: REMOVE CONTAMINATION CLUSTERS

In [30]:
# ============================================================================
# STEP 4: REMOVE CONTAMINATION CLUSTERS
# ============================================================================
print("\n" + "="*70 + "\nSTEP 4: REMOVE CONTAMINATION CLUSTERS\n" + "="*70)
n_before  = adata.n_obs
drop_mask = adata.obs[CELLTYPE_L3].astype(str).isin(DROP_CLUSTERS)
for c in DROP_CLUSTERS:
    n = (adata.obs[CELLTYPE_L3].astype(str)==c).sum()
    print(f"  {c}: {n:,}  [{annot_df.loc[c,'coarse_L3'] if c in annot_df.index else 'N/A'}]")
adata = adata[~drop_mask].copy()
print(f"{n_before:,} -> {adata.n_obs:,} cells  (removed {n_before-adata.n_obs:,})")
l3_str = adata.obs[CELLTYPE_L3].astype(str)
gc.collect()




STEP 4: REMOVE CONTAMINATION CLUSTERS
  Endothelia_vascular_Cap_a_c0: 265  [Interferon-activated Immune Contaminants]
  Endothelia_vascular_venous_systemic_c3: 2,764  [MHC-II-high APC-like Contaminants]
  Endothelia_vascular_venous_systemic_c5: 705  [Plasma/B-cell Contaminants]
  Fibro_myofibroblast_c0: 400  [Osteogenic Contaminants]
60,828 -> 56,694 cells  (removed 4,134)


6400

## Step 5: VERIFY DATA LAYERS

In [31]:
# ============================================================================
# STEP 5: VERIFY DATA LAYERS
# ============================================================================
print("\n" + "="*70 + "\nSTEP 5: VERIFY DATA LAYERS\n" + "="*70)
if 'counts' not in adata.layers:
    if adata.raw is not None:
        print("[WARN] 'counts' layer missing -- inferring from .raw.X")
        # Align to current adata.var_names to avoid shape mismatch
        missing = adata.var_names.difference(adata.raw.var_names)
        if len(missing) > 0:
            raise ValueError(f".raw is missing {len(missing)} current genes; cannot recover counts safely.")
        adata.layers['counts'] = sparse.csr_matrix(
            adata.raw[:, adata.var_names].X
        ).astype(np.float32)
    else:
        raise ValueError("No 'counts' layer and no .raw -- cannot continue.")
if 'log1p' not in adata.layers:
    adata.layers['log1p'] = adata.X.copy()
adata.X = adata.layers['log1p']
for lk in ['counts','log1p']:
    if lk in adata.layers and not sparse.issparse(adata.layers[lk]):
        adata.layers[lk] = sparse.csr_matrix(adata.layers[lk])
if not sparse.issparse(adata.X):
    adata.X = sparse.csr_matrix(adata.X)
print(f"[OK] shape={adata.shape}, counts={adata.layers['counts'].data.nbytes/1e9:.2f} GB")




STEP 5: VERIFY DATA LAYERS
[OK] shape=(56694, 36789), counts=0.98 GB


## Step 6: FILTER SMALL BATCHES

In [32]:
# ============================================================================
# STEP 6: FILTER SMALL BATCHES
# ============================================================================
print("\n" + "="*70 + "\nSTEP 6: FILTER SMALL BATCHES\n" + "="*70)
bc = adata.obs[BATCH_KEY].value_counts()
small = bc[bc < MIN_CELLS_BATCH].index.tolist()
if small:
    n_pre = adata.n_obs
    adata = adata[~adata.obs[BATCH_KEY].isin(small)].copy()
    print(f"Removed {len(small)} small batches: {n_pre:,} -> {adata.n_obs:,}")
else:
    print(f"[OK] All {adata.obs[BATCH_KEY].nunique()} batches pass minimum size")
gc.collect()




STEP 6: FILTER SMALL BATCHES
Removed 3 small batches: 56,694 -> 56,689


6332

## Step 7: HVG SELECTION + FULL-GENE .raw (shared memory, zero extra cost)

In [33]:
# ============================================================================
# STEP 7: HVG SELECTION + FULL-GENE .raw (shared memory, zero extra cost)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 7: HVG SELECTION + PRESERVE .raw\n" + "="*70)
print(f"Selecting {N_HVG} HVGs from {adata.n_vars:,} genes...")
try:
    sc.pp.highly_variable_genes(adata, layer='counts', n_top_genes=N_HVG,
                                 batch_key=BATCH_KEY, flavor='seurat_v3', subset=False)
    hvg_method = "batch-aware seurat_v3"
except Exception as e:
    print(f"  [WARN] {e}")
    try:
        sc.pp.highly_variable_genes(adata, layer='counts', n_top_genes=N_HVG,
                                     flavor='seurat_v3', subset=False)
        hvg_method = "seurat_v3 (no batch)"
    except Exception as e2:
        print(f"  [WARN] {e2}")
        sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, subset=False)
        hvg_method = "default"
print(f"HVG method: {hvg_method} | HVGs: {adata.var['highly_variable'].sum():,}")

# CRITICAL: preserve full genes to .raw BEFORE subsetting (shared memory, no copy)
print(f"Saving {adata.n_vars:,}-gene data to .raw (shared memory)...")
adata.raw = sc.AnnData(
    X   = adata.layers['counts'],   # shared memory -- no .copy()
    obs = adata.obs.copy(),
    var = adata.var.copy()
)
adata = adata[:, adata.var['highly_variable']].copy()

# FIX [4]: counts fallback after HVG subset must recover from .raw (full-gene counts),
# not from .X which holds log1p at this stage. Using .X here would silently feed
# log-transformed values into scVI as if they were raw counts.
if 'counts' not in adata.layers:
    print("[WARN] counts layer missing after HVG subset -- recovering from .raw")
    raw_hvg_counts = adata.raw[:, adata.var_names].X
    adata.layers['counts'] = sparse.csr_matrix(raw_hvg_counts).astype(np.float32)

print(f"HVG subset: {adata.shape} | .raw: {adata.raw.n_vars:,} genes (full)")
gc.collect()




STEP 7: HVG SELECTION + PRESERVE .raw
Selecting 4000 HVGs from 36,789 genes...
extracting highly variable genes
  [WARN] b'There are other near singularities as well. 0.090619\n'
extracting highly variable genes
HVG method: seurat_v3 (no batch) | HVGs: 4,000
Saving 36,789-gene data to .raw (shared memory)...
HVG subset: (56689, 4000) | .raw: 36,789 genes (full)


6441

## Step 8: scVI TRAINING (re-integration on cleaned data)

In [34]:
# ============================================================================
# STEP 8: scVI TRAINING (re-integration on cleaned data)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 8: scVI TRAINING\n" + "="*70)
SCVI_MODEL_PATH = MODEL_DIR / "scvi_stromal_v1"
scvi.model.SCVI.setup_anndata(adata, layer='counts', batch_key=BATCH_KEY)
model_scvi = scvi.model.SCVI(adata, n_latent=N_LATENT, n_hidden=N_HIDDEN,
    n_layers=N_LAYERS, dropout_rate=DROPOUT, gene_likelihood='nb', dispersion='gene-batch')
print(f"n_latent={N_LATENT}, cells={adata.n_obs:,}, HVGs={adata.n_vars:,}")

# scvi-tools >=1.x uses lightning accelerator/devices instead of use_gpu
train_accelerator = 'gpu' if USE_GPU else 'cpu'
train_devices = 1
print(f"Training backend: accelerator={train_accelerator}, devices={train_devices}")

t0 = time.time()
model_scvi.train(
    max_epochs=MAX_EPOCHS_SCVI,
    batch_size=BATCH_SIZE,
    early_stopping=True,
    early_stopping_patience=20,
    train_size=0.9,
    accelerator=train_accelerator,
    devices=train_devices,
    plan_kwargs={'lr': 1e-3}
 )
print(f"[OK] scVI done in {(time.time()-t0)/60:.1f} min")
model_scvi.save(str(SCVI_MODEL_PATH), overwrite=True)
pd.Series(adata.var_names.tolist()).to_csv(SCVI_MODEL_PATH/"hvg_genes.csv", index=False, header=False)
adata.obsm['X_scvi'] = model_scvi.get_latent_representation()
print(f"[OK] X_scvi: {adata.obsm['X_scvi'].shape}")



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



STEP 8: scVI TRAINING
n_latent=75, cells=56,689, HVGs=4,000
Training backend: accelerator=gpu, devices=1
Epoch 364/400:  91%|█████████ | 364/400 [35:51<03:32,  5.91s/it, v_num=1, train_loss_step=769, train_loss_epoch=748]
Monitored metric elbo_validation did not improve in the last 20 records. Best score: 753.023. Signaling Trainer to stop.
[OK] scVI done in 35.9 min
[OK] X_scvi: (56689, 75)


## Step 9: NEIGHBORS + UMAP (scVI latent)

In [38]:
# ============================================================================
# STEP 9: NEIGHBORS + UMAP (scVI latent)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 9: NEIGHBORS + UMAP (scVI)\n" + "="*70)
sc.pp.neighbors(adata, use_rep='X_scvi', n_neighbors=30,
                random_state=RANDOM_SEED, key_added='neighbors_scvi')
sc.tl.umap(adata, neighbors_key='neighbors_scvi', random_state=RANDOM_SEED)
adata.obsm['X_umap_scvi'] = adata.obsm['X_umap'].copy()
fig, ax = plt.subplots(figsize=(9,7))
sc.pl.embedding(adata, basis='umap', color=CELLTYPE_L2, ax=ax,
                show=False, frameon=False, size=2, legend_loc='right margin',
                title='Post-scVI UMAP (L2)')
fig.savefig(FIG_DIR/'scvi_umap_L2.pdf', dpi=300, bbox_inches='tight')
plt.close('all'); gc.collect()
print("[OK] Saved: scvi_umap_L2.pdf")




STEP 9: NEIGHBORS + UMAP (scVI)
computing neighbors
    finished (0:00:13)
computing UMAP
    finished (0:01:48)
[OK] Saved: scvi_umap_L2.pdf


## Step 10: BUILD scANVI LABELS (tissue-aware coarse_L3)

In [36]:
# ============================================================================
# STEP 10: BUILD scANVI LABELS (tissue-aware coarse_L3)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 10: BUILD scANVI TRAINING LABELS\n" + "="*70)

# Base: coarse_L3 for all retained cells
scanvi_labels = adata.obs['coarse_L3'].astype(str).copy()

# Tissue-aware Unknown assignment:
# Cells in pulmonary/alveolar clusters from non-lung/trachea tissue -> Unknown
# Rationale: aCap, alveolar fibroblasts, pulmonary pericytes etc. are
# lung-specific subtypes. Labeling nasal/sinus cells with these names
# would cause scANVI to learn spurious cross-tissue associations.
if TISSUE_KEY_AVAILABLE:
    tissue_str = adata.obs[TISSUE_KEY].astype(str)
    tissue_norm = (
        tissue_str.str.strip().str.lower()
        .str.replace(r'[_-]+', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
    )
    lung_trachea_norm = {
        x.strip().lower().replace('_', ' ').replace('-', ' ')
        for x in LUNG_TRACHEA_VALUES
    }
    exact_tissue_match = tissue_norm.isin(lung_trachea_norm)
    keyword_tissue_match = tissue_norm.str.contains(
        r'\blung\b|\btrachea\b|\bairway\b|\bbronch|\bbronchi|\bparenchyma\b|\bpulmon',
        regex=True,
        na=False,
    )
    is_lung_or_trachea = exact_tissue_match | keyword_tissue_match

    auto_extra = sorted(tissue_str[keyword_tissue_match & ~exact_tissue_match].unique().tolist())
    if auto_extra:
        print("[INFO] Additional tissue values treated as lung/trachea by keyword rules:")
        print(f"       {auto_extra[:20]}")

    is_pulm_alv = adata.obs[CELLTYPE_L3].astype(str).isin(PULMONARY_ALVEOLAR_CLUSTERS)
    tissue_mismatch = is_pulm_alv & ~is_lung_or_trachea
    n_mismatch = tissue_mismatch.sum()
    print(f"Lung/trachea cells:               {is_lung_or_trachea.sum():,}")
    print(f"Pulmonary/alveolar cluster cells: {is_pulm_alv.sum():,}")
    print(f"Tissue mismatch -> Unknown:       {n_mismatch:,}")
    if n_mismatch > 0:
        detail = (adata.obs.loc[tissue_mismatch, [CELLTYPE_L3, TISSUE_KEY]]
                  .value_counts().reset_index(name='n_cells'))
        print(f"\nMismatch detail:\n{detail.head(20).to_string(index=False)}")
    scanvi_labels[tissue_mismatch] = UNLABELED
else:
    print("[WARN] Tissue key unavailable -- skipping tissue-aware filtering.")

# FIX [2]: Vectorized kNN purity computation.
# The original Python for loop iterates over every cell individually,
# causing O(n) Python overhead that takes 30-60 min on 100k+ cells.
# NumPy broadcasting reduces this to a single matrix comparison.
print(f"\nComputing kNN purity (k={PURITY_K}) in scVI latent space...")
knn = NearestNeighbors(n_neighbors=PURITY_K, algorithm='auto', n_jobs=16)
knn.fit(adata.obsm['X_scvi'])
_, knn_idx = knn.kneighbors(adata.obsm['X_scvi'])
labels_arr = scanvi_labels.values                                  # (n_obs,)
neighbor_labels = labels_arr[knn_idx]                              # (n_obs, k)
# Fraction of k neighbors sharing the same label as the focal cell
purity = np.mean(neighbor_labels == labels_arr[:, None], axis=1).astype(np.float32)
# Unknown cells are not evaluated for purity; assign 1.0 to keep them as-is
is_unknown_mask = labels_arr == UNLABELED
purity[is_unknown_mask] = 1.0
adata.obs['label_purity_scanvi'] = purity

low_purity = (purity < PURITY_THRESHOLD) & ~is_unknown_mask
scanvi_labels[low_purity] = UNLABELED
print(f"  Low purity -> Unknown: {low_purity.sum():,} ({low_purity.sum()/adata.n_obs*100:.1f}%)")
del knn, knn_idx, neighbor_labels; gc.collect()

adata.obs['scanvi_label'] = scanvi_labels.values
n_labeled = (scanvi_labels != UNLABELED).sum()
n_unknown = (scanvi_labels == UNLABELED).sum()
print(f"\nLabel summary: {n_labeled:,} labeled | {n_unknown:,} Unknown")
print("Label distribution:")
for lbl, n in scanvi_labels.value_counts().head(25).items():
    print(f"  {lbl}: {n:,}{'  <-- UNLABELED' if lbl==UNLABELED else ''}")



STEP 10: BUILD scANVI TRAINING LABELS
[INFO] Additional tissue values treated as lung/trachea by keyword rules:
       ['lung parenchyma', 'respiratory airway']
Lung/trachea cells:               18,705
Pulmonary/alveolar cluster cells: 7,270
Tissue mismatch -> Unknown:       640

Mismatch detail:
__reconstructed_cluster_id tissue  n_cells
Muscle_smooth_pulmonary_c0   nose      417
Muscle_smooth_pulmonary_c1   nose      212
Muscle_smooth_pulmonary_c1  sinus        7
Muscle_smooth_pulmonary_c0  sinus        4

Computing kNN purity (k=30) in scVI latent space...
  Low purity -> Unknown: 23,587 (41.6%)

Label summary: 32,462 labeled | 24,227 Unknown
Label distribution:
  Unknown: 24,227  <-- UNLABELED
  Activated Antigen-presenting Venous Endothelia: 5,093
  Immune-recruiting (ACKR1+SELE+) Venous Endothelia: 4,240
  Angiogenic Venous Endothelia: 3,308
  Quiescent Lymphatic Endothelia: 2,996
  Perivascular Adventitial Fibroblasts: 2,064
  Homeostatic Alveolar Fibroblasts: 2,017
  Unresolve

## Step 11: scANVI TRAINING

In [37]:
# ============================================================================
# STEP 11: scANVI TRAINING
# ============================================================================
print("\n" + "="*70 + "\nSTEP 11: scANVI TRAINING\n" + "="*70)
SCANVI_MODEL_PATH = MODEL_DIR / "scanvi_stromal_v1"

# Guardrail: scANVI semi-supervised training requires >=2 labeled classes
_labels = adata.obs['scanvi_label'].astype(str)
_labeled_mask = _labels != UNLABELED
_n_labeled_classes = _labels[_labeled_mask].nunique()
print(f"Labeled classes (excluding '{UNLABELED}'): {_n_labeled_classes}")

if _n_labeled_classes < 2:
    print("[WARN] <2 labeled classes detected; skipping scANVI training and using scVI fallback outputs.")
    adata.obsm['X_scanvi'] = adata.obsm['X_scvi'].copy()
    adata.obs['cell_type_scanvi_pred'] = _labels.copy()
    adata.obs['scanvi_uncertainty'] = np.where(_labels == UNLABELED, 1.0, 0.0).astype(np.float32)
    print(f"[OK] Fallback X_scanvi: {adata.obsm['X_scanvi'].shape}")
    print("Fallback prediction distribution:")
    print(adata.obs['cell_type_scanvi_pred'].value_counts().head(25).to_string())
else:
    # FIX [1]: Remove the redundant SCANVI.setup_anndata call.
    # from_scvi_model inherits the data manager from the scVI model and handles
    # label registration internally. Calling setup_anndata beforehand creates a
    # conflicting second data manager that triggers AnnDataManagerValidationError
    # or silent overwrites in scvi-tools >= 1.0.
    # NOTE: n_samples_per_label is a TRAINING arg (in some versions), not a model-init arg.
    model_scanvi = scvi.model.SCANVI.from_scvi_model(
        model_scvi,
        unlabeled_category=UNLABELED,
        labels_key='scanvi_label'
    )

    # FIX [3]: Release scVI model from GPU and CPU memory before scANVI training.
    # model_scvi is no longer needed after from_scvi_model copies its weights.
    # Keeping it alive doubles GPU memory usage during scANVI training.
    del model_scvi
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"[OK] GPU cache cleared before scANVI training")

    # scvi-tools >=1.x uses lightning accelerator/devices instead of use_gpu
    train_accelerator = 'gpu' if USE_GPU else 'cpu'
    train_devices = 1
    print(f"Training backend: accelerator={train_accelerator}, devices={train_devices}")

    # Version-compatible handling of n_samples_per_label
    import inspect
    scanvi_train_kwargs = dict(
        max_epochs=MAX_EPOCHS_SCANVI,
        batch_size=BATCH_SIZE,
        early_stopping=True,
        early_stopping_patience=20,
        train_size=0.9,
        accelerator=train_accelerator,
        devices=train_devices,
    )
    if 'n_samples_per_label' in inspect.signature(model_scanvi.train).parameters:
        scanvi_train_kwargs['n_samples_per_label'] = 100

    t0 = time.time()
    model_scanvi.train(**scanvi_train_kwargs)
    print(f"[OK] scANVI done in {(time.time()-t0)/60:.1f} min")
    model_scanvi.save(str(SCANVI_MODEL_PATH), overwrite=True)
    adata.obsm['X_scanvi']             = model_scanvi.get_latent_representation()
    adata.obs['cell_type_scanvi_pred'] = model_scanvi.predict()
    soft_pred = model_scanvi.predict(soft=True)
    soft_pred_arr = soft_pred.to_numpy() if hasattr(soft_pred, 'to_numpy') else np.asarray(soft_pred)
    adata.obs['scanvi_uncertainty']    = (1 - soft_pred_arr.max(axis=1)).astype(np.float32)
    print(f"[OK] X_scanvi: {adata.obsm['X_scanvi'].shape}")
    print("scANVI prediction distribution:")
    print(adata.obs['cell_type_scanvi_pred'].value_counts().head(25).to_string())




STEP 11: scANVI TRAINING
Labeled classes (excluding 'Unknown'): 36
[OK] GPU cache cleared before scANVI training
Training backend: accelerator=gpu, devices=1
INFO     Training for 200 epochs.                                                                                  


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 200/200: 100%|██████████| 200/200 [43:21<00:00, 12.89s/it, v_num=1, train_loss_step=727, train_loss_epoch=742]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|██████████| 200/200 [43:21<00:00, 13.01s/it, v_num=1, train_loss_step=727, train_loss_epoch=742]
[OK] scANVI done in 43.4 min
[OK] X_scanvi: (56689, 75)
scANVI prediction distribution:
cell_type_scanvi_pred
Activated Antigen-presenting Venous Endothelia               6539
Immune-recruiting (ACKR1+SELE+) Venous Endothelia            5525
Angiogenic Venous Endothelia                                 4688
Unresolved Capillary Endothelia                              4473
Quiescent Lymphatic Endothelia                               3979
Perivascular Adventitial Fibroblasts                         3769
Homeostatic Alveolar Fibroblasts                             3534
Venous-like Lymphatic Endothelia                             3126
Niche-supporting Adventitial Fibroblasts                     3079
Interferon-primed Antigen-presenting Capillary Endothelia    2150
Lipogenic Peribronchial Fibroblasts                          1607
Quiescent Arterial Endothelia                  

## Step 12: UMAP + FIGURES (scANVI latent)

In [39]:
# ============================================================================
# STEP 12: UMAP + FIGURES (scANVI latent)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 12: UMAP (scANVI) + FIGURES\n" + "="*70)
sc.pp.neighbors(adata, use_rep='X_scanvi', n_neighbors=30,
                random_state=RANDOM_SEED, key_added='neighbors_scanvi')
sc.tl.umap(adata, neighbors_key='neighbors_scanvi', random_state=RANDOM_SEED)
adata.obsm['X_umap_scanvi'] = adata.obsm['X_umap'].copy()

fig, axes = plt.subplots(1, 3, figsize=(27, 8))
sc.pl.embedding(adata, basis='umap', color=CELLTYPE_L2, ax=axes[0],
                show=False, frameon=False, size=2, legend_loc='right margin',
                legend_fontsize=7, title='L2 Major Type')
sc.pl.embedding(adata, basis='umap', color='scanvi_label', ax=axes[1],
                show=False, frameon=False, size=2, legend_loc='right margin',
                legend_fontsize=6, title='scANVI Training Label (coarse_L3)')
sc.pl.embedding(adata, basis='umap', color='cell_type_scanvi_pred', ax=axes[2],
                show=False, frameon=False, size=2, legend_loc='right margin',
                legend_fontsize=6, title='scANVI Prediction')
plt.suptitle('Stromal/Vascular Post-scANVI', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(FIG_DIR/'scanvi_umap_overview.pdf', dpi=300, bbox_inches='tight')
plt.close('all'); print("[OK] Saved: scanvi_umap_overview.pdf")

fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.embedding(adata, basis='umap', color='scanvi_uncertainty',
                ax=ax, show=False, frameon=False, size=2,
                cmap='RdYlBu_r', vmin=0, vmax=0.5,
                title='scANVI Prediction Uncertainty')
fig.savefig(FIG_DIR/'scanvi_uncertainty_umap.pdf', dpi=300, bbox_inches='tight')
plt.close('all'); print("[OK] Saved: scanvi_uncertainty_umap.pdf")

if TISSUE_KEY_AVAILABLE:
    fig, ax = plt.subplots(figsize=(9, 7))
    sc.pl.embedding(adata, basis='umap', color=TISSUE_KEY, ax=ax,
                    show=False, frameon=False, size=2, title='Tissue Source')
    fig.savefig(FIG_DIR/'scanvi_umap_tissue.pdf', dpi=300, bbox_inches='tight')
    plt.close('all'); print("[OK] Saved: scanvi_umap_tissue.pdf")
gc.collect()




STEP 12: UMAP (scANVI) + FIGURES
computing neighbors
    finished (0:00:15)
computing UMAP
    finished (0:01:48)
[OK] Saved: scanvi_umap_overview.pdf
[OK] Saved: scanvi_uncertainty_umap.pdf
[OK] Saved: scanvi_umap_tissue.pdf


24044

## Step 13: LEIDEN CLUSTERING (scANVI latent)

In [40]:
# ============================================================================
# STEP 13: LEIDEN CLUSTERING (scANVI latent)
# ============================================================================
print("\n" + "="*70 + "\nSTEP 13: LEIDEN CLUSTERING (scANVI latent)\n" + "="*70)
for res in [0.4, 0.6, 0.8, 1.0]:
    key = f'leiden_scanvi_r{res:.1f}'
    sc.tl.leiden(adata, resolution=res, neighbors_key='neighbors_scanvi',
                 key_added=key, random_state=RANDOM_SEED)
    print(f"  Resolution {res}: {adata.obs[key].nunique()} clusters")




STEP 13: LEIDEN CLUSTERING (scANVI latent)
running Leiden clustering
    finished (0:00:28)
  Resolution 0.4: 15 clusters
running Leiden clustering
    finished (0:00:39)
  Resolution 0.6: 17 clusters
running Leiden clustering
    finished (0:00:57)
  Resolution 0.8: 20 clusters
running Leiden clustering
    finished (0:00:50)
  Resolution 1.0: 22 clusters


## Step 14: SAVE FINAL OUTPUT

In [41]:
# ============================================================================
# STEP 14: SAVE FINAL OUTPUT
# ============================================================================
print("\n" + "="*70 + "\nSTEP 14: SAVE OUTPUT\n" + "="*70)
adata.uns['stromal_reintegration_params'] = {
    'version':'1.1','date':'2026-03-03',
    'n_cells_input':int(n_before),'n_cells_output':int(adata.n_obs),
    'drop_clusters':DROP_CLUSTERS,
    'reassign_fibro':['Endothelia_vascular_venous_systemic_c4'],
    'reassign_alveolar':['Fibro_adventitial_c2'],
    'n_hvg':int(N_HVG),'hvg_method':hvg_method,'n_latent':N_LATENT,
    'batch_key':BATCH_KEY,'tissue_key':TISSUE_KEY,
    'celltype_l3_column_used':CELLTYPE_L3,
    'tissue_key_available':TISSUE_KEY_AVAILABLE,
    'lung_trachea_values':sorted(LUNG_TRACHEA_VALUES),
    'pulmonary_alveolar_clusters':PULMONARY_ALVEOLAR_CLUSTERS,
    'purity_k':PURITY_K,'purity_threshold':PURITY_THRESHOLD,
    'random_seed':RANDOM_SEED,
    'fixes':['scanvi_setup_anndata_removed','knn_purity_vectorized',
             'scvi_model_released_before_scanvi','counts_fallback_from_raw',
             'cluster_id_reconstructed_from_l2_plus_subcluster_id'],
}
print(f"Cells: {adata.n_obs:,} | HVGs: {adata.n_vars:,} | .raw: {adata.raw.n_vars:,}")
adata.write_h5ad(OUTPUT_H5AD, compression='gzip', compression_opts=9)
pd.Series(adata.var_names.tolist()).to_csv(OUTPUT_DIR/'hvg_genes_final.csv', index=False, header=False)
pd.Series(adata.raw.var_names.tolist()).to_csv(OUTPUT_DIR/'all_genes_raw.csv', index=False, header=False)
adata.obs[[CELLTYPE_L2,CELLTYPE_L3,'coarse_L3','annot_action','annot_confidence',
           'scanvi_label','label_purity_scanvi','cell_type_scanvi_pred',
           'scanvi_uncertainty']].to_csv(OUTPUT_DIR/'scanvi_label_summary.csv')
print(f"[OK] {OUTPUT_H5AD}")

elapsed = time.time()-PIPELINE_START
print(f"\n{'='*70}\nPIPELINE COMPLETE  ({elapsed/60:.1f} min)\n{'='*70}")
print(f"Cells: {n_before:,} -> {adata.n_obs:,}  |  "
      f"Labeled: {n_labeled:,}  |  Unknown: {n_unknown:,}")
print(f"\nNext steps:")
print(f"  1. Confirm TISSUE_KEY / LUNG_TRACHEA_VALUES match your data")
print(f"  2. Review scanvi_umap_overview.pdf -- label vs prediction agreement")
print(f"  3. Check scanvi_label_summary.csv for high-uncertainty REVIEW clusters")



STEP 14: SAVE OUTPUT
Cells: 56,689 | HVGs: 4,000 | .raw: 36,789


[OK] /home/h2048/data/py/0303/stromal_reintegration_v1/stromal_reintegrated_scvi_scanvi_v1_1.h5ad

PIPELINE COMPLETE  (502.3 min)
Cells: 60,828 -> 56,689  |  Labeled: 32,462  |  Unknown: 24,227

Next steps:
  1. Confirm TISSUE_KEY / LUNG_TRACHEA_VALUES match your data
  2. Review scanvi_umap_overview.pdf -- label vs prediction agreement
  3. Check scanvi_label_summary.csv for high-uncertainty REVIEW clusters
